# PCA Eigen-Portfolio + GenAI Factor Intelligence (v2)

This version upgrades the project with real statistical rigor on top of the original PCA + GenAI pipeline:

1. A wider universe (60 stocks, US + India) over a 2-year window instead of 20 stocks over 2 months.
2. A component stability check - does PC1 actually capture the same pattern if you fit PCA on a different time window, or is it noise?
3. A real out-of-sample backtest - does the eigen portfolio actually beat a naive equal-weighted portfolio on data it never saw during fitting?
4. The GenAI layer from before (LLM component interpretation, RAG-grounded anomaly explanations, automated report), now folding the stability and backtest results into the final report.

Runs in Google Colab. Needs a free Google AI Studio API key (aistudio.google.com/apikey) for the GenAI sections - no billing required.

In [ ]:
!pip -q install yfinance langchain langchain-google-genai langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers feedparser

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import time
import getpass
from sklearn.decomposition import PCA

## 1. Universe, data, and returns
60 large-cap stocks: 30 US (S&P 100-style) + 30 India (Nifty 50-style). Two years of daily data (2022-01-01 to 2024-03-01), with the last ~6 months held out as a genuine out-of-sample test set.

In [ ]:
us_tickers = ['AAPL','MSFT','GOOG','AMZN','META','NVDA','TSLA','JPM','V','JNJ',
              'WMT','PG','UNH','HD','MA','DIS','BAC','XOM','CVX','KO',
              'PEP','COST','ABBV','MRK','PFE','ADBE','CRM','NFLX','INTC','CSCO']

india_tickers = ['RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','ICICIBANK.NS','HINDUNILVR.NS',
                 'SBIN.NS','BHARTIARTL.NS','KOTAKBANK.NS','ITC.NS','LT.NS','AXISBANK.NS',
                 'ASIANPAINT.NS','MARUTI.NS','SUNPHARMA.NS','TITAN.NS','ULTRACEMCO.NS','WIPRO.NS',
                 'NESTLEIND.NS','POWERGRID.NS','NTPC.NS','BAJFINANCE.NS','BAJAJFINSV.NS','HCLTECH.NS',
                 'TATAMOTORS.NS','TATASTEEL.NS','COALINDIA.NS','ONGC.NS','ADANIENT.NS','GRASIM.NS']

name = us_tickers + india_tickers
print(f'Universe size: {len(name)} stocks ({len(us_tickers)} US + {len(india_tickers)} India)')

In [ ]:
import yfinance as yf

def dataset(name, start='2022-01-01', end='2024-03-01'):
    # Build each ticker as a DATE-INDEXED Series, then join on actual dates.
    # Mixing NYSE + NSE/BSE tickers means different holiday calendars, so each
    # ticker can have a different number of trading days - joining positionally
    # would silently misalign dates across markets, so we join on the real index instead.
    series_list = []
    for i in name:
        y = yf.Ticker(i)
        d = y.history(interval='1d', start=start, end=end)
        if d.empty:
            print(f'Skipping {i}: no data returned')
            continue
        d.index = d.index.tz_localize(None).normalize()
        intraday_ret = (d['Close'] - d['Open']) * 100 / d['Open']
        intraday_ret.name = i
        series_list.append(intraday_ret)
    y_f = pd.concat(series_list, axis=1, join='inner')  # keep only dates every market was open
    return y_f, y_f.index.tolist()

y_f, list_ = dataset(name)
print('Aligned trading days across all markets:', y_f.shape[0])
y_f.head()

In [ ]:
asset_returns = y_f.pct_change(1).dropna()
normed_returns = (asset_returns - asset_returns.mean()) / (asset_returns.std() + 1e-8)
normed_returns = normed_returns.dropna(axis=1)

train_end = '2023-09-01'  # ~1.75 years train, ~6 months held-out test
df_train = normed_returns[normed_returns.index <= train_end].copy()
df_test  = normed_returns[normed_returns.index > train_end].copy()
df_raw_train = asset_returns[asset_returns.index <= train_end].copy()
df_raw_test  = asset_returns[asset_returns.index > train_end].copy()

print('Train dataset:', df_train.shape)
print('Test dataset:', df_test.shape)
assert df_train.shape[0] > 0, 'df_train is empty - check date alignment above'

## 2. PCA - fit the eigen-portfolios (on TRAIN data only)

In [ ]:
pca = PCA(n_components=7)
fit = pca.fit_transform(df_train)
df_pca = pd.DataFrame(fit, columns=np.arange(pca.n_components_), index=df_train.index)

print('Variance explained per component:')
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {v:.1%}')
print(f'Cumulative: {pca.explained_variance_ratio_.sum():.1%}')
df_pca.head()

## 3. Component Stability Check

A PCA component is only meaningful if it represents a *real, persistent* market pattern - not something that only shows up because of the specific 2-year window we happened to choose. This check fits PCA on a series of rolling windows within the training period and measures how similar PC1 (and PC2) are from one window to the next, using cosine similarity between the eigenvectors. A value near 1.0 means the component is finding the same pattern every time; a value near 0 means it's essentially unstable/noisy.

In [ ]:
def rolling_pca_stability(data, window_days=90, step_days=30, n_components=7):
    dates = data.index
    windows = []
    start = 0
    while start + window_days <= len(dates):
        windows.append(data.iloc[start:start + window_days])
        start += step_days

    pc1_vectors, pc2_vectors, window_labels = [], [], []
    for w in windows:
        p = PCA(n_components=n_components)
        p.fit(w)
        pc1_vectors.append(p.components_[0])
        pc2_vectors.append(p.components_[1])
        window_labels.append(f'{w.index[0].date()} to {w.index[-1].date()}')

    def cosine_sim(a, b):
        # abs() because PCA eigenvectors can arbitrarily flip sign between fits -
        # a flipped-sign vector still represents the SAME underlying pattern.
        return abs(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    pc1_stability = [cosine_sim(pc1_vectors[i], pc1_vectors[i + 1]) for i in range(len(pc1_vectors) - 1)]
    pc2_stability = [cosine_sim(pc2_vectors[i], pc2_vectors[i + 1]) for i in range(len(pc2_vectors) - 1)]
    return window_labels, pc1_stability, pc2_stability

window_labels, pc1_stability, pc2_stability = rolling_pca_stability(df_train)

print('PC1 stability (cosine similarity between consecutive 90-day windows, stepping 30 days):')
for i, sim in enumerate(pc1_stability):
    print(f'  Window {i+1} -> {i+2}: {sim:.3f}')
print(f'  Average PC1 stability: {np.mean(pc1_stability):.3f}\n')

print('PC2 stability:')
for i, sim in enumerate(pc2_stability):
    print(f'  Window {i+1} -> {i+2}: {sim:.3f}')
print(f'  Average PC2 stability: {np.mean(pc2_stability):.3f}')

stability_summary = {
    'pc1_avg': float(np.mean(pc1_stability)),
    'pc2_avg': float(np.mean(pc2_stability)),
    'n_windows': len(window_labels)
}

## 4. Sharpe Ratio Helper

In [ ]:
def sharpe_ratio(ts_returns, periods_per_year=252):
    n_years = ts_returns.shape[0] / periods_per_year
    annualized_return = np.power(np.prod(1 + ts_returns), (1 / n_years)) - 1
    annualized_vol = ts_returns.std() * np.sqrt(periods_per_year)
    annualized_sharpe = annualized_return / annualized_vol
    return annualized_return, annualized_vol, annualized_sharpe

## 5. Out-of-Sample Backtest: Eigen Portfolio vs. Equal-Weighted

This is the real test: PCA never saw the test period. We take PC1's eigenvector (learned only from train data), turn it into portfolio weights (long/short, normalized so absolute weights sum to 1), and apply those *fixed* weights to the held-out test period's raw returns. We compare its Sharpe ratio against a naive equal-weighted portfolio over the exact same test period and universe. If the eigen portfolio doesn't beat (or at least compete with) the naive baseline out-of-sample, that's an honest, important finding - not a reason to hide the result.

In [ ]:
def eigenvector_to_weights(eigenvector):
    return eigenvector / np.sum(np.abs(eigenvector))

pc1_weights = eigenvector_to_weights(pca.components_[0])
weights_series = pd.Series(pc1_weights, index=df_train.columns)

# Apply TRAIN-derived weights to RAW (unstandardized) TEST-period returns
eigen_portfolio_test_returns = (df_raw_test[weights_series.index] * weights_series).sum(axis=1)

# Naive equal-weighted baseline over the same test period and universe
equal_weights = pd.Series(1 / len(df_raw_test.columns), index=df_raw_test.columns)
equal_weighted_test_returns = (df_raw_test * equal_weights).sum(axis=1)

eigen_ar, eigen_av, eigen_sharpe = sharpe_ratio(eigen_portfolio_test_returns)
equal_ar, equal_av, equal_sharpe = sharpe_ratio(equal_weighted_test_returns)

print('Out-of-sample backtest (test period, never seen during PCA fitting):')
print(f'  Eigen Portfolio 1:  return={eigen_ar:.2%}  vol={eigen_av:.2%}  sharpe={eigen_sharpe:.2f}')
print(f'  Equal-Weighted:     return={equal_ar:.2%}  vol={equal_av:.2%}  sharpe={equal_sharpe:.2f}')

backtest_summary = {
    'eigen_return': float(eigen_ar), 'eigen_vol': float(eigen_av), 'eigen_sharpe': float(eigen_sharpe),
    'equal_return': float(equal_ar), 'equal_vol': float(equal_av), 'equal_sharpe': float(equal_sharpe)
}

---
# GenAI Extensions

Each section follows the same four-part structure: **define the LLM -> define the pipeline -> build the chain -> run the chain.**

In [ ]:
GOOGLE_API_KEY = getpass.getpass('Enter your Google AI Studio API key: ')

## 6. LLM Component Interpreter

In [ ]:
# ============================================================
# 1. DEFINE THE LLM
# ============================================================
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash', temperature=0.3, google_api_key=GOOGLE_API_KEY)

def get_text(response):
    content = response.content
    if isinstance(content, list):
        return ''.join(block.get('text', '') for block in content if isinstance(block, dict))
    return content


# ============================================================
# 2. DEFINE THE PIPELINE (data prep + prompt template)
# ============================================================
def get_top_loadings(pca, feature_names, component_idx, n=5):
    loadings = pca.components_[component_idx]
    order = np.argsort(loadings)
    top_neg = [(feature_names[i], loadings[i]) for i in order[:n]]
    top_pos = [(feature_names[i], loadings[i]) for i in order[::-1][:n]]
    return top_pos, top_neg

interpret_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a quantitative portfolio analyst. You explain PCA-derived eigen-portfolios '
     'in clear, precise language for a research report. Given the top positively- and '
     'negatively-weighted stocks for a principal component, propose a short label (3-6 words) '
     'for the market factor it likely represents, then explain your reasoning in 2-3 sentences. '
     'Be specific about sectors/geographies when the tickers suggest a pattern. If unclear, say so '
     'honestly rather than overclaiming.'),
    ('human',
     'Component {idx} explains {variance:.1%} of variance.\n\n'
     'Top POSITIVE loadings:\n{pos}\n\nTop NEGATIVE loadings:\n{neg}\n\n'
     'Label and explain this component.')
])


# ============================================================
# 3. BUILD THE CHAIN
# ============================================================
interpret_chain = interpret_prompt | llm | RunnableLambda(get_text)


# ============================================================
# 4. RUN THE CHAIN
# ============================================================
feature_names = df_train.columns.tolist()
interpretations = {}
for i in range(pca.n_components_):
    pos, neg = get_top_loadings(pca, feature_names, i)
    pos_str = '\n'.join(f'  {t}: {w:+.3f}' for t, w in pos)
    neg_str = '\n'.join(f'  {t}: {w:+.3f}' for t, w in neg)
    try:
        interpretations[i] = interpret_chain.invoke({
            'idx': i + 1, 'variance': pca.explained_variance_ratio_[i],
            'pos': pos_str, 'neg': neg_str
        })
    except Exception as e:
        print(f'PC{i+1} failed: {e}')
        interpretations[i] = f'(interpretation unavailable: {e})'
    print(f'--- PC{i+1} ({pca.explained_variance_ratio_[i]:.1%} variance) ---')
    print(interpretations[i], '\n')
    time.sleep(4)  # stay under free-tier rate limits

## 7. RAG News Explainer

In [ ]:
# ============================================================
# 1. DEFINE THE LLM
# ============================================================
# Reuses `llm` and `get_text` from Section 6.


# ============================================================
# 2. DEFINE THE PIPELINE (retrieval: fetch -> chunk -> embed -> store)
# ============================================================
import feedparser
from urllib.parse import quote
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

def fetch_news_rss(query, max_items=5):
    url = f'https://news.google.com/rss/search?q={quote(query)}&hl=en-IN&gl=IN&ceid=IN:en'
    feed = feedparser.parse(url)
    return [f"{e.title}. {e.get('summary','')}" for e in feed.entries[:max_items]]

def build_retriever(raw_docs, k=4):
    chunks = splitter.create_documents(raw_docs)
    vs = FAISS.from_documents(chunks, embeddings)
    return vs.as_retriever(search_kwargs={'k': k})

rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a markets analyst writing a short factor-attribution note. Use ONLY the '
     'retrieved headlines given as context - do not invent facts. If the headlines do not '
     'clearly explain the move, say plainly that the cause is unclear from available news.'),
    ('human',
     'On {date}, eigen-portfolio Component {comp} moved sharply, driven mainly by: {top_stocks}.\n\n'
     'Retrieved headlines from around that date:\n{context}\n\n'
     'In 3-4 sentences, give a plausible explanation for this move, citing the headlines.')
])


# ============================================================
# 3. BUILD THE CHAIN
# ============================================================
rag_generation_chain = rag_prompt | llm | RunnableLambda(get_text)

def run_rag_chain(date, comp_idx, top_k=4):
    pos, neg = get_top_loadings(pca, feature_names, comp_idx, n=3)
    drivers = pos + neg
    top_stocks_str = ', '.join(t for t, _ in drivers)
    queries = [t.split('.')[0] for t, _ in drivers[:3]]

    raw_docs = []
    for q in queries:
        raw_docs += fetch_news_rss(f'{q} stock')
    if not raw_docs:
        return 'No relevant news retrieved for this date.'

    retriever = build_retriever(raw_docs, k=top_k)
    retrieved = retriever.invoke(f'{top_stocks_str} stock news')
    context = '\n'.join(f'- {d.page_content}' for d in retrieved)

    return rag_generation_chain.invoke({
        'date': date, 'comp': comp_idx + 1, 'top_stocks': top_stocks_str, 'context': context
    })


# ============================================================
# 4. RUN THE CHAIN
# ============================================================
def top_anomaly_dates(component_idx, top_n=2):
    col = df_pca.iloc[:, component_idx]
    return col.abs().sort_values(ascending=False).head(top_n).index.tolist()

anomaly_narratives = []
for comp_idx in range(3):
    for date in top_anomaly_dates(comp_idx, top_n=2):
        narrative = run_rag_chain(date, comp_idx)
        date_str = str(date.date()) if hasattr(date, 'date') else str(date)
        anomaly_narratives.append((date_str, comp_idx, narrative))
        print(f'--- PC{comp_idx+1} on {date_str} ---')
        print(narrative, '\n')
        time.sleep(4)

## 8. Auto Report Generation

In [ ]:
# ============================================================
# 1. DEFINE THE LLM
# ============================================================
# Reuses `llm` and `get_text` from Section 6.


# ============================================================
# 2. DEFINE THE PIPELINE (report assembly + summary prompt)
# ============================================================
def build_report_sections():
    sections = []
    sections.append(f'# Eigen-Portfolio Factor Report\nGenerated {datetime.date.today()}\n')
    sections.append(
        f'## Universe & Data\n{len(name)} stocks (US + India), '
        f'{df_train.shape[0]} training days, {df_test.shape[0]} held-out test days.\n')
    sections.append(
        f'## Variance Explained\nTop {pca.n_components_} components explain '
        f'{pca.explained_variance_ratio_.sum():.1%} of total variance in the universe.\n')

    sections.append(
        f'## Component Stability\nAverage PC1 cosine similarity across rolling windows: '
        f"{stability_summary['pc1_avg']:.3f}. Average PC2: {stability_summary['pc2_avg']:.3f} "
        f"(across {stability_summary['n_windows']} windows). Values near 1.0 indicate a stable, "
        f'persistent factor; values near 0 indicate an unstable/noisy one.\n')

    sections.append(
        f"## Out-of-Sample Backtest\nEigen Portfolio 1: return={backtest_summary['eigen_return']:.2%}, "
        f"vol={backtest_summary['eigen_vol']:.2%}, Sharpe={backtest_summary['eigen_sharpe']:.2f}. "
        f"Equal-Weighted baseline: return={backtest_summary['equal_return']:.2%}, "
        f"vol={backtest_summary['equal_vol']:.2%}, Sharpe={backtest_summary['equal_sharpe']:.2f}.\n")

    sections.append('## Component Interpretations\n')
    for i in range(pca.n_components_):
        sections.append(f'### PC{i+1} ({pca.explained_variance_ratio_[i]:.1%} variance)\n{interpretations[i]}\n')

    sections.append('## Anomaly Explanations (RAG-grounded)\n')
    for date, comp, narrative in anomaly_narratives:
        sections.append(f'**{date} — PC{comp+1}:** {narrative}\n')

    return sections

exec_summary_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You write concise executive summaries for quant research reports.'),
    ('human', 'Summarize this factor report in one tight paragraph for a portfolio manager, '
              'including whether the backtest and stability results support taking the eigen '
              'portfolio seriously:\n\n{report}')
])


# ============================================================
# 3. BUILD THE CHAIN
# ============================================================
summary_chain = exec_summary_prompt | llm | RunnableLambda(get_text)


# ============================================================
# 4. RUN THE CHAIN
# ============================================================
report_sections = build_report_sections()
exec_summary = summary_chain.invoke({'report': '\n'.join(report_sections)})

final_report = f'# Executive Summary\n{exec_summary}\n\n' + '\n'.join(report_sections)

with open('eigen_portfolio_report.md', 'w') as f:
    f.write(final_report)

print(final_report[:1500], '...\n\n[full report saved to eigen_portfolio_report.md]')

### Optional: download the report from Colab
```python
from google.colab import files
files.download('eigen_portfolio_report.md')
```